In [0]:
# 1. Kiểm tra danh sách các file trong Volume
files = dbutils.fs.ls("/Volumes/workspace/bronze/raw_data")


print(f"Tổng số file tìm thấy: {len(files)}")
for f in files:
    print(f"- {f.name} ({f.size / 1024:.2f} KB)")


In [0]:
import pyspark.sql.functions as F

# 1. Đọc file Stores.csv thô
stores_path = "/Volumes/workspace/bronze/raw_data/Stores.csv"

df_stores_raw = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(stores_path))

# 2. CHUẨN HÓA TÊN CỘT: Xóa khoảng trắng, đổi thành snake_case (square_meters, open_date)
for col_name in df_stores_raw.columns:
    clean_name = col_name.strip().lower().replace(" ", "_")
    df_stores_raw = df_stores_raw.withColumnRenamed(col_name, clean_name)

# 3. Chuẩn hóa tầng Bronze: Thêm audit metadata bằng chuẩn Unity Catalog
df_stores_bronze = (df_stores_raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path")))

# 4. Ghi trực tiếp ra Delta Table trong Unity Catalog
(df_stores_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.bronze_stores"))

print(" ĐÃ TẠO THÀNH CÔNG BẢNG DELTA: workspace.bronze.bronze_stores!")

# 5. Hiển thị bảng
display(spark.table("workspace.bronze.bronze_stores").limit(5))


In [0]:
%sql
-- Xem lịch sử của bảng Delta: Ai tạo, lúc nào, phiên bản nào
DESCRIBE HISTORY workspace.bronze.bronze_stores


In [0]:
import datetime
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
import pyspark.sql.functions as F

# Dùng đúng chuẩn DateType: datetime.date(năm, tháng, ngày)
updates_data = [
    (1, "Australia", "Australian Capital Territory", 850, datetime.date(2008, 1, 1)),
    (999, "Vietnam", "Ha Noi Flagship", 1200, datetime.date(2026, 10, 10))
]

schema = StructType([
    StructField("storekey", IntegerType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True),
    StructField("square_meters", IntegerType(), True),
    StructField("open_date", DateType(), True)
])

df_updates = (spark.createDataFrame(updates_data, schema)
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit("erp_incremental_update.csv")))

df_updates.createOrReplaceTempView("stores_incoming_updates")
print(" Đã chuẩn hóa đúng kiểu DateType theo chuẩn Delta Lake!")


In [0]:
%sql
MERGE INTO workspace.bronze.bronze_stores AS target
USING stores_incoming_updates AS source
ON target.storekey = source.storekey

-- NẾU ĐÃ CÓ: Cập nhật diện tích mới và thời gian nạp
WHEN MATCHED THEN
  UPDATE SET 
    target.square_meters = source.square_meters,
    target._ingested_at = source._ingested_at

-- NẾU CHƯA CÓ: Thêm mới cửa hàng vào bảng
WHEN NOT MATCHED THEN
  INSERT *;


In [0]:
%sql
MERGE INTO workspace.bronze.bronze_stores AS target
USING stores_incoming_updates AS source
ON target.storekey = source.storekey

-- NẾU ĐÃ CÓ: Cập nhật diện tích mới và thời gian nạp
WHEN MATCHED THEN
  UPDATE SET 
    target.square_meters = source.square_meters,
    target._ingested_at = source._ingested_at

-- NẾU CHƯA CÓ: Thêm mới cửa hàng vào bảng
WHEN NOT MATCHED THEN
  INSERT *;


In [0]:
%sql
SELECT * FROM workspace.bronze.bronze_stores WHERE storekey IN (1, 999)


In [0]:
%sql
-- 1. Quay ngược thời gian về Version 0 (Thời điểm ban đầu: Store 1 vẫn là 595 m² và CHƯA CÓ Store 999)
SELECT * FROM workspace.bronze.bronze_stores VERSION AS OF 0 WHERE storekey IN (1, 999)


In [0]:
%sql
-- 2. Xem lịch sử biến đổi của bảng (Commit Log)
DESCRIBE HISTORY workspace.bronze.bronze_stores


In [0]:
%sql
-- Gom các file nhỏ thành file lớn chuẩn mực để tăng tốc độ truy vấn
OPTIMIZE workspace.bronze.bronze_stores


In [0]:
import pyspark.sql.functions as F

# Danh sách 4 file còn lại cần nạp vào Bronze
tables_to_ingest = {
    "Products.csv": "bronze_products",
    "Customers.csv": "bronze_customers",
    "Sales.csv": "bronze_sales",
    "Exchange_Rates.csv": "bronze_exchange_rates"
}

volume_base_path = "/Volumes/workspace/bronze/raw_data"

for file_name, table_name in tables_to_ingest.items():
    file_path = f"{volume_base_path}/{file_name}"
    print(f"Đang xử lý nạp {file_name} -> {table_name}...")
    
    # 1. Đọc CSV thô
    df = (spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(file_path))
    
    # 2. Tự động chuẩn hóa toàn bộ tên cột sang snake_case (xóa khoảng trắng)
    for col in df.columns:
        clean_col = col.strip().lower().replace(" ", "_")
        df = df.withColumnRenamed(col, clean_col)
        
    # 3. Thêm Audit Metadata tầng Bronze
    df = (df
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path")))
    
    # 4. Ghi ra bảng Delta trong Unity Catalog
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.bronze.{table_name}"))
    
    row_count = spark.table(f"workspace.bronze.{table_name}").count()
    print(f" HOÀN TẤT: workspace.bronze.{table_name} ({row_count:,} dòng)\n")

print(" TẤT CẢ DỮ LIỆU TẦNG BRONZE ĐÃ ĐƯỢC NẠP XONG 100%!")
